# Évaluer les alertes de hausse

## Ce qu’on appelle une alerte ici

Le notebook part des prévisions du backtest et considère qu’un événement réel est une hausse d’au moins **15 %** entre le prix de référence et le prix cible. Il vérifie ensuite si les séries signalées contiennent effectivement davantage de ces fortes hausses.

La **précision** répond à : « parmi les alertes émises, quelle part correspond à une forte hausse ? » Le **rappel** répond à : « parmi toutes les fortes hausses, quelle part a été signalée ? » Une bonne précision seule ne suffit donc pas si beaucoup d’événements passent inaperçus.

Les mois identifiés comme figés sont écartés de plusieurs comparaisons pour limiter leur effet. Ces résultats décrivent un backtest historique et ne garantissent pas les performances futures.

## 1. Précision et rappel des alertes à 2 et 3 mois

On compare les alertes du modèle aux signaux de momentum, pour plusieurs proportions de séries retenues parmi les plus fortes hausses prévues. Le taux de base indique à quelle fréquence l’événement de +15 % se produit sans filtrage.

In [1]:
import sys; sys.path.append("..")
from src.evaluation.baselines import load_panel
from src.models.lgbm import run_backtest
from src.alerts.rules import add_changes, evaluate_alerts

brut = load_panel(brut=True)
res = add_changes(run_backtest(brut, brut))

for h in (2, 3):
    print(evaluate_alerts(res, h, event=0.15).to_string(index=False), "\n")

h=2 | hausse réelle >= 15% | fréquence de base : 16.1% | n=11786
          règle  alertes_%  alertes  précision_%  rappel_%  lift
         modèle          5      590         47.8      14.9  2.97
         modèle         10     1179         39.1      24.3  2.43
         modèle         20     2358         32.2      40.1  2.00
momentum 3 mois          5      590         10.7       3.3  0.66
momentum 3 mois         10     1179          9.9       6.2  0.62
momentum 3 mois         20     2358         10.5      13.1  0.65 

h=3 | hausse réelle >= 15% | fréquence de base : 21.6% | n=11497
          règle  alertes_%  alertes  précision_%  rappel_%  lift
         modèle          5      575         58.1      13.4  2.69
         modèle         10     1150         49.0      22.7  2.27
         modèle         20     2300         42.5      39.4  1.97
momentum 3 mois          5      575         14.1       3.3  0.65
momentum 3 mois         10     1152         13.7       6.4  0.63
momentum 3 mois        

## 2. Comparer le modèle à une règle de rebond

Cette expérience teste une règle simple : après une baisse récente, le prix pourrait rebondir. On compare les séries les mieux classées par cette règle à celles classées par le modèle, avec la même définition d’une hausse réelle.

In [2]:
r = res[(res.h == 3) & ~res.bad].dropna(subset=["d3"])
real = r.true_chg >= 0.15
for name, sig in [("modèle", r.pred_chg), ("rebond (-momentum)", -r.mom_chg)]:
    for q in (0.05, 0.10):
        a = sig >= sig.quantile(1 - q)
        prec = (a & real).sum() / a.sum()
        print(f"{name:20s} top {int(q*100):2d}% | précision {100*prec:.1f}% | lift {prec/real.mean():.2f}")

modèle               top  5% | précision 58.1% | lift 2.69
modèle               top 10% | précision 49.0% | lift 2.27
rebond (-momentum)   top  5% | précision 49.0% | lift 2.27
rebond (-momentum)   top 10% | précision 44.7% | lift 2.07


## 3. Les alertes sont-elles régulières selon l’année ?

Le tableau mesure, année par année, la fréquence des hausses réelles, le nombre d’alertes et leur précision. Il aide à voir si la moyenne masque une année où le système fonctionne moins bien.

In [4]:
import pandas as pd
q = 0.10

r = res[(res.h == 3) & ~res.bad].dropna(subset=["d3"]).copy()
r["real"] = r.true_chg >= 0.15
r["alert"] = r.pred_chg >= r.pred_chg.quantile(1 - q)

g = r.groupby("year")
print(pd.DataFrame({"hausses_réelles_%": 100 * g.real.mean(),
                    "alertes": g.alert.sum(),
                    "précision_%": 100 * r[r.alert].groupby("year").real.mean()}).round(1))

print(r.groupby(r.date.dt.month)[["real", "alert"]].mean().round(2))   # saisonnalité

      hausses_réelles_%  alertes  précision_%
year                                         
2022               26.7      405         55.8
2023               18.5      250         35.2
2024               38.0       11         81.8
2025               19.1       79         57.0
2026               18.2      405         48.4
      real  alert
date             
1     0.26   0.18
2     0.27   0.15
3     0.28   0.10
4     0.30   0.07
5     0.25   0.03
6     0.20   0.01
7     0.18   0.03
8     0.17   0.02
9     0.11   0.04
10    0.13   0.13
11    0.14   0.21
12    0.15   0.21


## 4. Seuils de hausse utilisés pour les alertes

Les quantiles affichés servent à traduire les prédictions du modèle en seuils de vigilance. Ils sont estimés à partir des résultats historiques et décrivent les hausses prévues dans ce backtest.

In [5]:
for h in (2, 3):
    r = res[res.h == h].dropna(subset=["d3"])
    print(f"h={h}", (100 * r.pred_chg.quantile([0.80, 0.90, 0.95])).round(1).to_dict())

h=2 {0.8: 5.6, 0.9: 9.1, 0.95: 13.9}
h=3 {0.8: 8.0, 0.9: 12.6, 0.95: 18.6}


## 5. Couverture récente des relevés

On regarde la part de prix inchangés dans les mois récents et le nombre de séries au dernier mois disponible. Une prévision ne peut être produite pour une série si son historique récent est absent.

In [6]:
p = pd.read_csv("../data/processed/panel_long_brut.csv", parse_dates=["date"])
print(p[p.date >= "2026-04-01"].groupby("date")["stale"].mean().round(2).to_string())
print(p[p.date == p.date.max()].shape[0], "séries au dernier mois")

date
2026-04-01    0.12
2026-05-01    0.10
2026-06-01    0.14
2026-07-01    0.19
2026-08-01    0.39
2026-09-01    0.27
238 séries au dernier mois


## 6. Séries disponibles et alertes courantes

La dernière cellule inspecte les dates de fin des historiques et les alertes du fichier de prévisions courant. C’est un contrôle de couverture : une série absente du dernier mois ne doit pas être interprétée comme une absence de hausse.

In [7]:
# 1. Combien de séries s'arrêtent avant septembre 2026 ?
p = pd.read_csv("../data/processed/panel_long_brut.csv", parse_dates=["date"])
dern = p.groupby(["market", "commodity"])["date"].max()
print(dern.value_counts().sort_index().tail(8))

# 2. Les 9 alertes
f = pd.read_csv("../data/processed/previsions.csv")
print(f[f.alerte.notna()][["market", "commodity", "h", "prix_actuel", "prix_prevu", "hausse_prevue"]]
        .round(2).to_string(index=False))

date
2026-02-01      2
2026-03-01     31
2026-04-01     10
2026-05-01      3
2026-06-01     17
2026-07-01     12
2026-08-01      6
2026-09-01    238
Name: count, dtype: int64
      market     commodity  h  prix_actuel  prix_prevu  hausse_prevue
   Korbongou Maize (white)  2        131.0      141.98           0.08
Ogaro centre        Millet  2        205.0      218.73           0.07
      Tsévié   Beans (red)  2        258.0      279.09           0.08
 Vogan Gbeke Maize (white)  2        134.0      144.55           0.08
   Korbongou Maize (white)  3        131.0      145.98           0.11
     Kpalimé Maize (white)  3        136.0      150.05           0.10
Ogaro centre        Millet  3        205.0      225.12           0.10
      Tsévié   Beans (red)  3        258.0      288.10           0.12
 Vogan Gbeke Maize (white)  3        134.0      150.21           0.12
